# Chapter 18 — Building Data Applications with Streamlit

Streamlit turns Python scripts into interactive data applications. Learn the execution model, input widgets, caching, charts, data loading, secrets, and deployment considerations.

## Prerequisites
Python, Pandas, and at least one visualization library; install Streamlit separately.

## Learning goals
- Explain Streamlit’s script rerun model.
- Build layouts with titles, columns, containers, and sidebar widgets.
- Use caching appropriately for data and resources.
- Connect CSV and MySQL data safely.
- Structure, test, and deploy a small analytical application.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Execution model and display API
2. Layout and widgets
3. Caching and session state
4. Data sources and secrets
5. Application architecture and deployment

### Execution model and display API
A Streamlit app is a Python script run by `streamlit run app.py`. Widget interactions trigger a script rerun from top to bottom. Display functions include `st.title`, `header`, `subheader`, `write`, `markdown`, `dataframe`, `table`, `metric`, and chart methods such as `line_chart` and `plotly_chart`. Use a clear top-to-bottom flow and avoid expensive unguarded work on every rerun.

#### Demonstration 1 — Install and check Streamlit

Run the code and inspect both the output and the assumptions behind it.

In [2]:
try:
 import streamlit as st
 print("Streamlit version:",st.__version__)
except ImportError:
 print("Install with: pip install streamlit")

Install with: pip install streamlit


### Layout and widgets
`st.sidebar`, `st.columns`, `st.container`, `st.expander`, and `st.tabs` organize content. Widgets include `selectbox`, `multiselect`, `slider`, `date_input`, `text_input`, `number_input`, `checkbox`, `radio`, `file_uploader`, and `button`. Most return the current selection/value; `key` gives a stable identity. Widget state is managed by Streamlit and can be explicitly read through `st.session_state`.

#### Demonstration 2 — Create a minimal app script

Run the code and inspect both the output and the assumptions behind it.

In [3]:
from pathlib import Path
app_code = r'''import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(page_title="Sales Explorer", layout="wide")
st.title("Sales Explorer")
DATA = Path(__file__).resolve().parents[1] / "data" / "sales.csv"
@st.cache_data
def load_data(path):
    return pd.read_csv(path, parse_dates=["order_date"])
sales = load_data(DATA)
regions = st.sidebar.multiselect("Region", sorted(sales.region.unique()), default=sorted(sales.region.unique()))
filtered = sales[sales.region.isin(regions)]
c1,c2,c3 = st.columns(3)
c1.metric("Orders", f"{len(filtered):,}")
c2.metric("Revenue", f"{filtered.revenue.sum():,.0f}")
c3.metric("Profit", f"{filtered.profit.sum():,.0f}")
st.subheader("Revenue by region")
st.bar_chart(filtered.groupby("region").revenue.sum())
st.dataframe(filtered, use_container_width=True)
'''
Path("app.py").write_text(app_code,encoding="utf-8"); print("Created app.py; run: streamlit run app.py")

Created app.py; run: streamlit run app.py


### Caching and session state
`@st.cache_data` caches serializable data results and returns copies; `@st.cache_resource` caches shared resources such as database connections or ML models. Configure TTL, max entries, and hash behavior deliberately. `st.session_state` persists values across reruns for a user session. Caching must not expose one user’s private data to another; include user/security context in cache keys where relevant.

#### Demonstration 3 — Demonstrate filtering logic outside Streamlit

Run the code and inspect both the output and the assumptions behind it.

In [4]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd
sales=pd.read_csv(DATA/"sales.csv"); selected=["North","West"]
filtered=sales[sales.region.isin(selected)]
print("Orders:",len(filtered),"Revenue:",round(filtered.revenue.sum(),2))

Data directory: /mnt/data/python_data_analysis_book/data
Orders: 119 Revenue: 20532456.34


### Data sources and secrets
Use Pandas readers for uploaded CSV/Excel files. Database credentials belong in Streamlit secrets (`.streamlit/secrets.toml`) or environment variables, never committed source files. Use parameterized SQL, read-only credentials for dashboards, connection timeouts, and explicit error handling. Treat uploaded files as untrusted input and validate size, schema, and content.

#### Demonstration 4 — Demonstrate cached function pattern

Run the code and inspect both the output and the assumptions behind it.

In [5]:
try:
 import streamlit as st
 @st.cache_data(ttl=300)
 def summarize(frame): return frame.groupby("region").revenue.sum()
 print("Inside Streamlit, call summarize(dataframe) to cache this pure calculation.")
except ImportError: print("Install Streamlit to run cache decorators: pip install streamlit")

Install Streamlit to run cache decorators: pip install streamlit


### Application architecture and deployment
Separate loading, cleaning, metric calculation, and display functions. Use stable schemas and cache expensive read-only queries with a suitable TTL. Test filtering edge cases, missing data, empty results, and large inputs. Deploy with pinned dependencies, secrets management, resource limits, and a refresh strategy. The app script itself is the application; notebook cells are a learning environment.

#### Demonstration 5 — Show a safe database pattern

Run the code and inspect both the output and the assumptions behind it.

In [6]:
database_example = r'''import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
url = URL.create("mysql+mysqlconnector", username=os.environ["MYSQL_USER"], password=os.environ["MYSQL_PASSWORD"], host=os.getenv("MYSQL_HOST","localhost"), port=int(os.getenv("MYSQL_PORT","3306")), database=os.environ["MYSQL_DATABASE"])
engine = create_engine(url, pool_pre_ping=True)
with engine.connect() as conn:
    result = pd.read_sql(text("SELECT region, SUM(revenue) AS revenue FROM orders GROUP BY region"), conn)
'''
print(database_example)

import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
url = URL.create("mysql+mysqlconnector", username=os.environ["MYSQL_USER"], password=os.environ["MYSQL_PASSWORD"], host=os.getenv("MYSQL_HOST","localhost"), port=int(os.getenv("MYSQL_PORT","3306")), database=os.environ["MYSQL_DATABASE"])
engine = create_engine(url, pool_pre_ping=True)
with engine.connect() as conn:
    result = pd.read_sql(text("SELECT region, SUM(revenue) AS revenue FROM orders GROUP BY region"), conn)



## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Execution model and display API.
- Explain one important concept from: Layout and widgets.
- Explain one important concept from: Caching and session state.
- Explain one important concept from: Data sources and secrets.
- Explain one important concept from: Application architecture and deployment.